# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/7ayder-99/flyrank_internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb

import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/7ayder-99/flyrank_internship"
REPO_DIR = "flyrank_internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import duckdb
con = duckdb.connect()
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")
print("Ready:", os.getcwd())

Ready: /content/flyrank_internship


In [3]:
from huggingface_hub import HfApi

api = HfApi()
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset", token=hf_token)
for f in files:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [4]:
DIM_CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"
con.sql(f"DESCRIBE SELECT * FROM {DIM_CONTENT}").show()
con.sql(f"SELECT * FROM {DIM_CONTENT} LIMIT 5").show()

┌────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│        column_name         │ column_type │  null   │   key   │ default │  extra  │
│          varchar           │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_hash_id             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ url_hash_id                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_char_count         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_token_count        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ url_char_count             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ content_created_date       │ DATE        │ YES     │ NULL    │ 

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

A page is worth flagging for review if it used to attract real search traffic, hasn't been touched in a long time, and is currently losing impressions. Volume alone isn't enough — a stale page with no traffic isn't worth anyone's time, and a fresh page that's declining might just need more time to mature.

The signals it leans on (verified below with real bucket checks, not assumed):

Staleness — days_since_update, derived from dim_content.content_updated_date. This is the signal behind FlyRank's refresh flags.
Volume — impressions_mar, the current month's search impressions. This is the signal behind the quick-win flag (a page needs enough visibility to be worth acting on).
(Position/CTR is checked separately as a second, independent signal — not folded into this particular rule's score, since the rule is about refresh priority, not CTR repair.)

The score:

stale    = days_since_update >= 180
visible  = impressions_feb >= 100          # had real traffic historically
declining = impressions_mar < impressions_feb

score = stale * visible * declining * impressions_feb

Reads as: only pages that are stale AND used to be visible AND are currently declining get a non-zero score — and among those, the ones that used to get more traffic rank higher (bigger opportunity cost from letting them keep declining).

Reason codes (one per scored row, explaining why it scored):

stale_visible_declining — met all three conditions, scored > 0
not_stale — fails the staleness check alone
not_visible_historically — fails the volume check alone
not_declining — fails the decline check alone (traffic stable or growing)

Action label:

score > 0 → review_for_refresh
score == 0 → monitorر

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
REL_FEB   = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet')"
REL_MARCH = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')"
DIM_CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"

# aggregate each month to one row per page
page_month_sql = """
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions,
           SUM(gsc_clicks) AS clicks,
           AVG(gsc_avg_position) AS avg_position
    FROM {rel}
    GROUP BY client_hash_id, content_hash_id
"""

feb_df = con.sql(page_month_sql.format(rel=REL_FEB)).df()
mar_df = con.sql(page_month_sql.format(rel=REL_MARCH)).df()

pages = mar_df.merge(feb_df, on=["client_hash_id", "content_hash_id"],
                      suffixes=("_mar", "_feb"), how="inner")
print("Pages present in both months:", len(pages))
pages.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages present in both months: 303572


,client_hash_id,content_hash_id,impressions_mar,clicks_mar,avg_position_mar,impressions_feb,clicks_feb,avg_position_feb
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,4.394234,601.0,3.0,4.527386
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,0.0,2.714744,207.0,1.0,2.991600
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,6.481453,156.0,1.0,8.661331
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,6.320337,1567.0,7.0,5.662970
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,4.459107,2176.0,4.0,3.732656


In [6]:
import pandas as pd

# --- Signal 1: Staleness (behind FlyRank's refresh flags) ---

content = con.sql(f"""
    SELECT client_hash_id, content_hash_id, content_updated_date
    FROM {DIM_CONTENT}
""").df()

pages = pages.merge(content, on=["client_hash_id", "content_hash_id"], how="left")

ANALYSIS_DATE = pd.Timestamp("2026-03-31")
pages["content_updated_date"] = pd.to_datetime(pages["content_updated_date"])
pages["days_since_update"] = (ANALYSIS_DATE - pages["content_updated_date"]).dt.days

pages["staleness_tier"] = pd.cut(
    pages["days_since_update"],
    bins=[-1, 30, 90, 180, 365, 1_000_000],
    labels=["<30d", "30-90d", "90-180d", "180-365d", "365d+"]
)

pages["is_declining"] = (pages["impressions_mar"] < pages["impressions_feb"]).astype(int)

signal1 = pages.groupby("staleness_tier", observed=True).agg(
    n=("is_declining", "size"),
    decline_rate=("is_declining", "mean")
)
print("=== Signal 1: Staleness vs decline rate ===")
print(signal1)

# --- Signal 2: CTR vs Position (behind CTR-fix logic) ---

pages["ctr"] = pages["clicks_mar"] / pages["impressions_mar"].replace(0, pd.NA)

pages["position_tier"] = pd.cut(
    pages["avg_position_mar"],
    bins=[0, 3, 10, 20, 50, 1000],
    labels=["1-3", "4-10", "11-20", "21-50", "50+"]
)

signal2 = pages.groupby("position_tier", observed=True).agg(
    n=("ctr", "size"),
    mean_ctr=("ctr", "mean")
)
print("\n=== Signal 2: Position tier vs mean CTR ===")
print(signal2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Signal 1: Staleness vs decline rate ===
                    n  decline_rate
staleness_tier                     
<30d              964      0.619295
30-90d          29680      0.401887
90-180d          3608      0.151608
180-365d         3816      0.039832

=== Signal 2: Position tier vs mean CTR ===
                   n  mean_ctr
position_tier                 
1-3            14216  0.011384
4-10           74488  0.004998
11-20          29771  0.003185
21-50          30594  0.002281
50+            11208  0.000905


In [7]:
# ============================================
# Section 2: Build the ranked queue
# ============================================

import os
import pandas as pd

# --- Thresholds ---
STALE_DAYS = 180
MIN_HISTORICAL_IMPRESSIONS = 100

# --- Signals ---
pages["stale"] = (
    pages["days_since_update"] >= STALE_DAYS
).astype(int)

pages["visible"] = (
    pages["impressions_feb"] >= MIN_HISTORICAL_IMPRESSIONS
).astype(int)

pages["declining"] = (
    pages["impressions_mar"] < pages["impressions_feb"]
).astype(int)

# --- Score ---
# Only stale + historically visible + declining pages get a score.
# Higher historical impressions = higher opportunity cost.
pages["action_score"] = (
    pages["stale"]
    * pages["visible"]
    * pages["declining"]
    * pages["impressions_feb"]
)

# --- Reason code ---
pages["reason_code"] = "not_stale"

pages.loc[
    (pages["stale"] == 1) &
    (pages["visible"] == 0),
    "reason_code"
] = "not_visible_historically"

pages.loc[
    (pages["stale"] == 1) &
    (pages["visible"] == 1) &
    (pages["declining"] == 0),
    "reason_code"
] = "not_declining"

pages.loc[
    pages["action_score"] > 0,
    "reason_code"
] = "stale_visible_declining"

# --- Action ---
pages["action"] = pages["action_score"].gt(0).map({
    True: "review_for_refresh",
    False: "monitor"
})

# --- Rank ---
pages = pages.sort_values(
    ["action_score", "impressions_feb"],
    ascending=[False, False]
).reset_index(drop=True)

pages["rank"] = range(1, len(pages) + 1)

# --- Select queue columns ---
queue = pages[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "impressions_feb",
        "impressions_mar",
        "days_since_update",
        "action_score",
        "reason_code",
        "action"
    ]
].copy()

# --- Write CSV ---
output_path = "work/outputs/baseline_action_score.csv"

os.makedirs(os.path.dirname(output_path), exist_ok=True)

queue.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Rows: {len(queue):,}")
print(f"Review for refresh: {(queue['action'] == 'review_for_refresh').sum():,}")
print(f"Monitor: {(queue['action'] == 'monitor').sum():,}")

print("\n=== Top 20 ranked pages ===")
print(queue.head(20))

Saved: work/outputs/baseline_action_score.csv
Rows: 303,572
Review for refresh: 19
Monitor: 303,553

=== Top 20 ranked pages ===
    rank           client_hash_id           content_hash_id  impressions_feb  \
0      1  client_c182d11e4862a37d  content_3af16a3c5dffb146           1912.0   
1      2  client_c182d11e4862a37d  content_715dfb7ac57fc2f3           1130.0   
2      3  client_c182d11e4862a37d  content_5c9e0961371c7f2d            887.0   
3      4  client_c182d11e4862a37d  content_84d0bb5aaf292517            790.0   
4      5  client_c182d11e4862a37d  content_70537a712a8d554e            543.0   
5      6  client_c182d11e4862a37d  content_454deefc8b3da983            450.0   
6      7  client_c182d11e4862a37d  content_a99d75e3bf98772e            398.0   
7      8  client_c182d11e4862a37d  content_34b8de016805d44d            319.0   
8      9  client_c182d11e4862a37d  content_9dc017d4ef83c0d9            276.0   
9     10  client_c182d11e4862a37d  content_b37c25257ed39a1c            

The ranked queue applies the baseline action rule to every page in the merged dataset.

Each page receives an action_score based on four conditions:

Stale: days_since_update >= 180
Historically visible: impressions_feb >= 100
Declining: impressions_mar < impressions_feb
Score: historical February impressions for pages meeting all three conditions.

Pages that satisfy all three signals receive the review_for_refresh action and are ranked by their action_score. All other pages receive the monitor action.

The resulting queue contains 303,572 pages. Only 19 pages qualify for refresh review, while 303,553 pages are assigned to monitoring.

The output was written to:

work/outputs/baseline_action_score.csv

This creates the baseline ranked queue that will be used for the next section.

# 3. Top-20 review
For each of the top 20: action, reason code, confidence note, and what would make it wrong.



The top-20 queue was reviewed individually using the baseline action, reason code, and underlying signals.

For the first 19 pages, the recommendation is `review_for_refresh` because each page is stale (at least 180 days since update), had meaningful historical visibility (at least 100 February impressions), and declined in March.

The 20th page is included in the ranked top-20 output but receives `monitor` because it does not meet the staleness threshold.

| Rank | Action             | Reason code             | Confidence note                                                                                                                         | What would make it wrong                                                                                       |
| ---: | ------------------ | ----------------------- | --------------------------------------------------------------------------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------------------- |
|    1 | review_for_refresh | stale_visible_declining | High — all three baseline signals are satisfied.                                                                                        | Incorrect update date, unreliable impression data, or a temporary/intentional traffic drop.                    |
|    2 | review_for_refresh | stale_visible_declining | High — all three baseline signals are satisfied.                                                                                        | Incorrect update date, unreliable impression data, or a temporary/intentional traffic drop.                    |
|    3 | review_for_refresh | stale_visible_declining | High — all three baseline signals are satisfied.                                                                                        | Incorrect update date, unreliable impression data, or a temporary/intentional traffic drop.                    |
|    4 | review_for_refresh | stale_visible_declining | High — all three baseline signals are satisfied.                                                                                        | Incorrect update date, unreliable impression data, or a temporary/intentional traffic drop.                    |
|    5 | review_for_refresh | stale_visible_declining | High — all three baseline signals are satisfied.                                                                                        | Incorrect update date, unreliable impression data, or a temporary/intentional traffic drop.                    |
|    6 | review_for_refresh | stale_visible_declining | High — all three baseline signals are satisfied.                                                                                        | Incorrect update date, unreliable impression data, or a temporary/intentional traffic drop.                    |
|    7 | review_for_refresh | stale_visible_declining | High — all three baseline signals are satisfied.                                                                                        | Incorrect update date, unreliable impression data, or a temporary/intentional traffic drop.                    |
|    8 | review_for_refresh | stale_visible_declining | High — all three baseline signals are satisfied.                                                                                        | Incorrect update date, unreliable impression data, or a temporary/intentional traffic drop.                    |
|    9 | review_for_refresh | stale_visible_declining | High — all three baseline signals are satisfied.                                                                                        | Incorrect update date, unreliable impression data, or a temporary/intentional traffic drop.                    |
|   10 | review_for_refresh | stale_visible_declining | High — all three baseline signals are satisfied.                                                                                        | Incorrect update date, unreliable impression data, or a temporary/intentional traffic drop.                    |
|   11 | review_for_refresh | stale_visible_declining | High — all three baseline signals are satisfied.                                                                                        | Incorrect update date, unreliable impression data, or a temporary/intentional traffic drop.                    |
|   12 | review_for_refresh | stale_visible_declining | High — all three baseline signals are satisfied.                                                                                        | Incorrect update date, unreliable impression data, or a temporary/intentional traffic drop.                    |
|   13 | review_for_refresh | stale_visible_declining | High — all three baseline signals are satisfied.                                                                                        | Incorrect update date, unreliable impression data, or a temporary/intentional traffic drop.                    |
|   14 | review_for_refresh | stale_visible_declining | High — all three baseline signals are satisfied.                                                                                        | Incorrect update date, unreliable impression data, or a temporary/intentional traffic drop.                    |
|   15 | review_for_refresh | stale_visible_declining | High — all three baseline signals are satisfied.                                                                                        | Incorrect update date, unreliable impression data, or a temporary/intentional traffic drop.                    |
|   16 | review_for_refresh | stale_visible_declining | High — all three baseline signals are satisfied.                                                                                        | Incorrect update date, unreliable impression data, or a temporary/intentional traffic drop.                    |
|   17 | review_for_refresh | stale_visible_declining | High — all three baseline signals are satisfied.                                                                                        | Incorrect update date, unreliable impression data, or a temporary/intentional traffic drop.                    |
|   18 | review_for_refresh | stale_visible_declining | High — all three baseline signals are satisfied.                                                                                        | Incorrect update date, unreliable impression data, or a temporary/intentional traffic drop.                    |
|   19 | review_for_refresh | stale_visible_declining | High — all three baseline signals are satisfied.                                                                                        | Incorrect update date, unreliable impression data, or a temporary/intentional traffic drop.                    |
|   20 | monitor            | not_stale               | High — the page has substantial visibility but is only  -83 days from the analysis date, so it correctly fails the staleness condition. | The content date could be incorrect, or the baseline staleness threshold may be too strict for this page type. |

### Review summary

The first 19 pages are strong matches for the **baseline rule** because their scores are non-zero and each satisfies the three required conditions. However, the confidence reflects confidence in the **rule application**, not proof that refreshing the page will recover traffic.

The main factors that could make any recommendation wrong are inaccurate content-update dates, unreliable impression measurements, seasonality, changes in search demand, technical SEO issues, intentional content retirement, or a decline caused by factors that a content refresh would not address.


## 4. Weak picks + leakage check
Which picks look wrong and why? Confirm no product flags or future windows leaked in.




The main weakness in the current queue is the **staleness signal**.

The Signal 1 bucket check showed the opposite relationship from the original assumption: pages updated more recently had higher decline rates, while pages in the `180-365d` bucket had the lowest observed decline rate. Therefore, the `days_since_update >= 180` condition is not empirically supported as a predictor of decline in this dataset.

As a result, the 19 `review_for_refresh` picks should be treated as **baseline candidates rather than high-confidence recommendations**. Their scores are valid under the defined rule, but the rule itself may be selecting pages based on staleness even though staleness was not associated with higher decline in the validation check.

A second potential weakness is **data quality around content dates**. For example, rank 20 has `days_since_update = -83`, which means its recorded update date is later than the analysis date. This does not affect its current `monitor` label, but it suggests that content dates should be validated before using staleness as a production decision signal.

### Leakage check

No future-window leakage was introduced into the baseline score.

The scoring rule uses:

* `days_since_update`, calculated relative to the fixed analysis date of **2026-03-31**
* `impressions_feb` as the historical visibility signal
* `impressions_mar` as the current-month decline signal

No data from a period after March 2026 is used to calculate the action score.

There are also **no product flags or product-related signals** included in the baseline scoring logic. The score is based only on content freshness, historical impressions, and the February-to-March impression change.

Therefore, the baseline queue is **free of future-window and product-flag leakage based on the fields used in the scoring code**.

### Verdict

The queue is technically valid as a baseline implementation, but the **refresh recommendations should be considered low-confidence from a causal/ predictive perspective** because the staleness signal was contradicted by the bucket validation.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.